# Lakebase 101 Demo — Bootstrap Delta Source Tables

Run this notebook **before** `databricks bundle deploy`.

Creates:
1. Catalog + schema in Unity Catalog
2. `products` (12 rows) — product catalog
3. `customers_directory` (50k) — realistic customer profiles via Faker
4. `customer_360_gold` (50k) — enriched gold-layer profiles
5. `sales_events` (5M) — transactional sales data

All tables have **Change Data Feed** enabled for synced table support.

In [0]:
# Match these to your databricks.yml variables
CATALOG = "lakebase101_demo"
SCHEMA = "dais_demo"
FQN = f"{CATALOG}.{SCHEMA}"

# Row counts
NUM_CUSTOMERS = 50_000
NUM_SALES_EVENTS = 5_000_000

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS lakebase101_demo;
CREATE SCHEMA IF NOT EXISTS lakebase101_demo.dais_demo;

In [0]:
%pip install faker --quiet
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

CATALOG = "lakebase101_demo"
SCHEMA = "dais_demo"
FQN = f"{CATALOG}.{SCHEMA}"

products_data = [
    (1, "Wireless Headphones", "Electronics", 79.99),
    (2, "Mechanical Keyboard", "Electronics", 149.99),
    (3, "USB-C Hub", "Electronics", 49.99),
    (4, "Standing Desk", "Furniture", 599.99),
    (5, "Ergonomic Chair", "Furniture", 449.99),
    (6, "Monitor Arm", "Furniture", 89.99),
    (7, "Noise-Canceling Earbuds", "Audio", 199.99),
    (8, "Portable Speaker", "Audio", 69.99),
    (9, "Webcam HD", "Peripherals", 99.99),
    (10, "Laptop Stand", "Peripherals", 39.99),
    (11, "Desk Lamp", "Accessories", 34.99),
    (12, "Cable Organizer", "Accessories", 19.99),
]

df_products = spark.createDataFrame(products_data, ["product_id", "product_name", "category", "price"])
df_products.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQN}.products")
spark.sql(f"ALTER TABLE {FQN}.products SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
print(f"✅ {FQN}.products: {df_products.count()} rows")

In [0]:
from faker import Faker
import random

fake = Faker()
Faker.seed(42)
random.seed(42)

NUM_CUSTOMERS = 50_000
segments = ["VIP", "Loyal", "Regular", "At-Risk", "New"]
cities = [
    "San Francisco", "New York", "Chicago", "Austin", "Seattle",
    "Denver", "Boston", "Portland", "Miami", "Atlanta",
    "Dallas", "Phoenix", "Philadelphia", "San Diego", "Nashville",
    "Minneapolis", "Detroit", "Orlando", "Charlotte", "Raleigh",
]

customers = []
for i in range(1, NUM_CUSTOMERS + 1):
    customers.append((
        i,
        fake.name(),
        random.choice(segments),
        random.choice(cities),
        round(random.uniform(100, 15000), 2),
        random.randint(1, 80),
    ))

schema = StructType([
    StructField("customer_id", IntegerType()),
    StructField("full_name", StringType()),
    StructField("segment", StringType()),
    StructField("city", StringType()),
    StructField("total_sales", DecimalType(12, 2)),
    StructField("num_sales", IntegerType()),
])

df_customers = spark.createDataFrame(customers, schema)
df_customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQN}.customers_directory")
spark.sql(f"ALTER TABLE {FQN}.customers_directory SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
print(f"✅ {FQN}.customers_directory: {NUM_CUSTOMERS} rows")

In [0]:
from datetime import date, timedelta
import random

random.seed(99)

df_cust = spark.table(f"{FQN}.customers_directory")
df_prod = spark.table(f"{FQN}.products")

# Build gold layer: enrich customers with churn scoring + product recommendations
df_gold = (
    df_cust
    .withColumn("email", F.concat(F.lower(F.regexp_replace(F.col("full_name"), " ", ".")), F.lit("@example.com")))
    .withColumn("lifetime_value", F.col("total_sales"))
    .withColumn("total_orders", F.col("num_sales"))
    .withColumn("avg_order_value", F.round(F.col("total_sales") / F.greatest(F.col("num_sales"), F.lit(1)), 2))
    .withColumn("churn_risk_score", F.round((F.col("customer_id") * 17 % 100) / F.lit(100.0), 2))
    .withColumn("churn_risk_band",
        F.when(F.col("churn_risk_score") < 0.33, "Low")
         .when(F.col("churn_risk_score") < 0.66, "Medium")
         .otherwise("High"))
    .withColumn("recommended_product_id", (F.col("customer_id") % 12) + 1)
    .withColumn("last_order_date", F.date_sub(F.current_date(), (F.col("customer_id") % 90).cast("int")))
    .withColumn("gold_updated_at", F.current_timestamp())
)

# Join product recommendations
df_gold = (
    df_gold
    .join(df_prod.select(
        F.col("product_id").alias("recommended_product_id"),
        F.col("product_name").alias("recommended_product_name"),
        F.col("category").alias("recommended_category"),
        F.col("price").alias("recommended_price")
    ), on="recommended_product_id", how="left")
    .select(
        "customer_id", "full_name", "email", "city", "segment",
        "lifetime_value", "total_orders", "avg_order_value",
        "churn_risk_score", "churn_risk_band",
        "recommended_product_id", "recommended_product_name",
        "recommended_category", "recommended_price",
        "last_order_date", "gold_updated_at"
    )
)

df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQN}.customer_360_gold")
spark.sql(f"ALTER TABLE {FQN}.customer_360_gold SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
print(f"✅ {FQN}.customer_360_gold: {df_gold.count()} rows")

In [0]:
from pyspark.sql import functions as F

# Generate 5M sales events using Spark (fast, no Faker needed for transactions)
df_events = (
    spark.range(1, 5_000_001)
    .withColumnRenamed("id", "event_id")
    .withColumn("customer_id", (F.col("event_id") % NUM_CUSTOMERS) + 1)
    .withColumn("product_id", (F.col("event_id") % 12) + 1)
    .withColumn("amount", F.round(F.lit(20.0) + (F.col("event_id") * 37 % 500), 2))
    .withColumn("event_date", F.date_sub(F.current_date(), (F.col("event_id") % 365).cast("int")))
)

df_events.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQN}.sales_events")
spark.sql(f"ALTER TABLE {FQN}.sales_events SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
print(f"✅ {FQN}.sales_events: {df_events.count()} rows")

In [0]:
print(f"""
✅ Bootstrap complete!

Tables created in {FQN}:
  - products          (12 rows)
  - customers_directory (50k rows)
  - customer_360_gold   (50k rows)
  - sales_events        (5M rows)

All tables have Change Data Feed enabled.
You can now run: databricks bundle deploy -t presenter
""")